In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import UEG_response as ur

In [ ]:
# Units
hbar = 1.0
aB = 1.0
m = 1.0
e = 1.0

# Tolerances
reltol = 1e-16
abstol = 1e-8
eta_log  = 1e-6
eta_sqrt = 1e-6
eta_pol = 1e-4
points_n = 5
tol_upper = 1e-8
dx = 1e-4
lower = 1e-6
limit = 50



In [ ]:
# Temperature scan.
# Conditions
rs = 3.23
thetas = np.logspace(-2, np.log10(500), 100)


# Input
z1 = 0.0
y1 = 1.0
z2 = 0.1
y2 = 1.5
csTheta = 0.4


classical_chi2 = np.zeros(shape=thetas.shape, dtype=complex)
quantum_chi2   = np.zeros(shape=thetas.shape, dtype=complex)
norm_quantum   = np.zeros(shape=thetas.shape)
norm_classical = np.zeros(shape=thetas.shape)
for i, theta in enumerate(thetas):
    # Normalisation
    qF = (9*np.pi/4)**(1/3) / (rs*aB)
    EF = hbar**2 * qF**2 / (2*m)
    beta = 1/(theta*EF)
    n = 3/(4*np.pi*rs**3)
    beta_eff = beta / np.sqrt(1 + (1/theta)**2 )

    norm_quantum[i]   = n*beta_eff**2
    norm_classical[i] = n*beta**2

    # Physical units
    k1     = y1 * qF
    omega1 = z1  / (beta*hbar)
    k2     = y2 * qF
    omega2 = z2  / (beta*hbar)

    classical_chi2[i] = ur.classical_ideal_quadratic_response(k1, omega1, k2, omega2, csTheta, n, beta, m, dc=dx, eta_pol=eta_pol, reltol=reltol, abstol=abstol)[0]

    # Physical units
    k1     = y1 * qF
    omega1 = z1 / (beta_eff*hbar)
    k2     = y2 * qF
    omega2 = z2 / (beta_eff*hbar)

    quantum_chi2[i] =  ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta,
                                                    m, hbar, n=n, beta=beta, ms=2,
                                                    reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                                    dx=dx, points_n=points_n, force_output=True)[0]

# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
n = 3/(4*np.pi*rs**3)

# Physical units
k1     = y1 * qF
omega1 = z1 * EF / hbar
k2     = y2 * qF
omega2 = z2 * EF / hbar
ground_state_chi2_0 = ur.ground_state_ideal_quadratic_response(omega1, k1, omega2, k2, csTheta, m, hbar, n, ms=2) * np.ones(shape=thetas.shape)

norm_ground_state = (n/EF**2) * np.ones(shape=thetas.shape)


plt.plot(thetas, np.real(quantum_chi2)/norm_quantum,   '-k', label=r"Quantum ($\beta_x = \beta_{eff}$)")
plt.plot(thetas, np.imag(quantum_chi2)/norm_quantum,   '-r')

plt.xscale('log')
plt.yscale('log')

yrange = plt.ylim()

plt.plot(thetas, np.real(classical_chi2)/norm_classical, '--k', label=r"Classical ($\beta_x = \beta$)")
plt.plot(thetas, np.imag(classical_chi2)/norm_classical, '--r')

plt.plot(thetas, np.real(ground_state_chi2_0)/norm_ground_state, ':k', label=r"$T = 0$ ($\beta_x = E_F^{-1}$)")
plt.plot(thetas, np.imag(ground_state_chi2_0)/norm_ground_state, ':r')


plt.xlim([np.min(thetas), np.max(thetas)])
# plt.ylim(yrange)

plt.legend()

plt.xlabel(r"$\Theta$")
plt.ylabel(r"$\chi^{(2)}_{0}(\vec{k}_1, \omega_1, \vec{k}_2, \omega_2)$ [$n\beta_{x}^2$]")

plt.text(2e-2, 2e-1, r"$\cos\theta = %g$"%(csTheta))

plt.ylim([1e-2, 1e0])
# plt.savefig(f"figures/limits_temperature.jpg", dpi=400, bbox_inches='tight')


In [ ]:
# Large k-asymptotes
# Conditions
rs = 3.23
theta = 1.0


# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
beta = 1/(theta*EF)
n = 3/(4*np.pi*rs**3)
beta_eff = beta / np.sqrt(1 + (1/theta)**2 )

norm = n*beta**2

# Limit as k1 -> 0
omega1 = 0.0 / (beta_eff*hbar)
omega2 = 0.0 / (beta_eff*hbar)
k2 = 2.0 * qF
csTheta = 0.4

k1 = np.logspace(-1, np.log10(50), 100) * qF


chi0_2 =  ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta,
                                        m, hbar, n=n, beta=beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True)

E0 = hbar**2*k1**2/(2*m)
chi0_2_approx = - ur.ideal_linear_response(omega2, k2, m, hbar, n, beta, ms=2, 
                                           reltol=reltol, abstol=abstol, 
                                           eta_log=eta_log, tol_upper=tol_upper, points_n=points_n, force_output=True) / E0

chi0_2_0 =  ur.ideal_quadratic_response(omega1, 0.0, omega2, k2, csTheta,
                                        m, hbar, n=n, beta=beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True) * np.ones(shape=k1.shape)


plt.plot(k1/qF, np.real(chi0_2)/norm, '-k', label=r'$|\vec{k}_2| = %.1f\,q_F$'%(k2/qF))
plt.plot(k1/qF, np.real(chi0_2_approx)/norm, '--k')
plt.plot(k1/qF, np.real(chi0_2_0)/norm, ':k')

# Limit as k1 -> 0 and k2 -> 0
omega1 = 0.0 / (beta_eff*hbar)
omega2 = 0.0 / (beta_eff*hbar)
csTheta = 0.4
s = 1.3


k2 = s * k1


chi0_2 =  ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta,
                                        m, hbar, n=n, beta=beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True)

E1 = hbar**2*k1**2/(2*m)
E2 = hbar**2*k2**2/(2*m)
E12 = hbar**2*(k1**2 + 2*csTheta*k1*k2 + k2**2)/(2*m)
chi0_2_approx = n * (E1 + E2 + E12)/(E1 * E2 * E12)

chi0_2_0 =  ur.ideal_quadratic_response(omega1, 0.0, omega2, 0.0, csTheta,
                                        m, hbar, n=n, beta=beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True) * np.ones(shape=k1.shape)


plt.plot(k1/qF, np.real(chi0_2)/norm, '-r', label=r'$|\vec{k}_2| = %.2f\,|\vec{k}_1|$'%(s))
plt.plot(k1/qF, np.real(chi0_2_approx)/norm, '--r')
plt.plot(k1/qF, np.real(chi0_2_0)/norm, ':r')

plt.xscale('log')
plt.yscale('log')

plt.xlabel(r"$|\vec{k}_1|/q_F$")
plt.ylabel(r"$\chi_0^{(2)}(\vec{k}_1, \vec{k}_2)$ [$n\beta_{eff}^2$]")
plt.legend()
plt.text(1.5e-1, 1e-3, r"$\cos\theta = %.1f$"%(csTheta))

plt.ylim([1e-4, 1e0])
plt.xlim([np.min(k1/qF), 20])

# plt.savefig(f"figures/limits_k.jpg", dpi=400, bbox_inches='tight')
